# 08 — 2026 Campaign Case Study: Metadata Fragility vs the Graph Operator


## 1. Imports, config, and the case-study package lists

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, time
from pathlib import Path
import numpy as np, pandas as pd
import scipy.sparse as sp
from scipy.sparse import csgraph
from scipy.sparse.csgraph import dijkstra
import networkx as nx
from networkx.algorithms.community import louvain_communities
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
ALPHA = 0.2
CUT = pd.Timestamp("2026-01-01")
ROOT = Path("..")
ECO = os.environ.get("ECO", "npm")

META_FEATURES = [
    "name_exist","name_length","dist_tags_exist","dist_tags_length","versions_exist",
    "versions_length","versions_num_count","maintainers_exist","description_exist",
    "description_length","readme_exist","readme_length","author_exist","author_name",
    "author_email","license_exist","license_length","keywords_exist","keywords_length",
    "keywords_num_count","homepage_exist","homepage_length","github_exist","github_length",
    "bugslink_exist","bugslink_length","issueslink_exist","issueslink_length",
    "dependencies_exist","dependencies_length","scripts_exist","scripts_length",
    "devDependencies_exist","devDependencies_length","directories_exist",
    "directories_length","package_age_days","package_modified_duration_days",
    "package_published_duration_days","author_CPN","author_service_time","author_CCS",
    "maintainer_CPN","maintainer_service_time","maintainer_CCS","contributor_CPN",
    "contributor_service_time","contributor_CCS","publisher_CPN","publisher_service_time",
    "publisher_CCS","pull_request","issues","fork_number","star","subscriber_count",
]

CASE = {
    "npm": ["axios","react-hook-form","hls.js","@mistralai/mistralai","node-ipc",
            "@tanstack/react-router","@mastra/core","echarts-for-react","@antv/g6",
            "@opensearch-project/opensearch"],
    "pypi": ["litellm","lightning","mistralai","guardrails-ai"],
}

## 2. Graph substrate helpers (Warfield levels, transitive reach)

In [2]:
def build_graph_structures(src_arr, tgt_arr, n_nodes):
    A = sp.csr_matrix((np.ones(len(src_arr), np.int8), (src_arr, tgt_arr)), shape=(n_nodes, n_nodes))
    n_scc, scc = csgraph.connected_components(A, directed=True, connection="strong")
    cs, ct = scc[src_arr], scc[tgt_arr]; keep = cs != ct
    dag_edges = (np.unique(np.stack([cs[keep], ct[keep]], axis=1), axis=0) if keep.any()
                 else np.empty((0, 2), np.int64))
    ones = np.ones(len(dag_edges), np.int8)
    dep_csr = sp.csr_matrix((ones, (dag_edges[:, 0], dag_edges[:, 1])), shape=(n_scc, n_scc))
    inf_csr = sp.csr_matrix((ones, (dag_edges[:, 1], dag_edges[:, 0])), shape=(n_scc, n_scc))
    indeg = np.diff(dep_csr.indptr).astype(np.int64).copy()
    inf_ip, inf_ix = inf_csr.indptr, inf_csr.indices
    queue = [int(u) for u in np.flatnonzero(indeg == 0)]; topo, head = [], 0
    while head < len(queue):
        u = queue[head]; head += 1; topo.append(u)
        for v in inf_ix[inf_ip[u]:inf_ip[u + 1]]:
            indeg[v] -= 1
            if indeg[v] == 0: queue.append(int(v))
    assert len(topo) == n_scc, "condensation is not a DAG"
    lev = np.zeros(n_scc, np.int32)
    for u in topo:
        vs = inf_ix[inf_ip[u]:inf_ip[u + 1]]
        if len(vs): lev[vs] = np.maximum(lev[vs], lev[u] + 1)
    return {"n_scc": n_scc, "scc": scc, "dag_edges": dag_edges,
            "dep_csr": dep_csr, "inf_csr": inf_csr, "topo": np.array(topo), "lev": lev}


def reach_weighted_sums(succ_indptr, succ_indices, order, weights, slab_bits=8192, row_chunk=4096):
    n, k = weights.shape; w32 = np.ascontiguousarray(weights, np.float32); sums = np.zeros((n, k))
    order_nz = [int(u) for u in order if succ_indptr[u + 1] > succ_indptr[u]]
    for c0 in range(0, n, slab_bits):
        c1 = min(c0 + slab_bits, n); nbits = c1 - c0
        reach = np.zeros((n, (nbits + 7) // 8), np.uint8)
        js = np.arange(c0, c1); reach[js, (js - c0) >> 3] = (128 >> ((js - c0) & 7)).astype(np.uint8)
        for u in order_nz:
            a, b = succ_indptr[u], succ_indptr[u + 1]; reach[u] |= np.bitwise_or.reduce(reach[succ_indices[a:b]], axis=0)
        for r0 in range(0, n, row_chunk):
            r1 = min(r0 + row_chunk, n)
            sums[r0:r1] += np.unpackbits(reach[r0:r1], axis=1, count=nbits).astype(np.float32) @ w32[c0:c1]
    return sums

## 3. Per-ecosystem prospective scoring + greedy evasion

In [3]:
def run_eco(eco):
    t0 = time.time()
    DATA_DIR = ROOT / "data" / "input" / eco
    OSV_DIR = DATA_DIR
    df = pd.read_csv(DATA_DIR / "all_features.csv", encoding="utf-8", low_memory=False)
    df = df.drop_duplicates("package", keep="first")
    df = df[df["status"] == "ok"].reset_index(drop=True)
    if "bugslink_exist" in df.columns:
        df["issueslink_exist"] = df["bugslink_exist"]; df["issueslink_length"] = df["bugslink_length"]
    N = len(df)
    label_arr = df["label"].astype(int).to_numpy()
    y = ((label_arr == 1) | (label_arr == -1)).astype(np.int64)     # risky (nb08 positive class)
    m_comp, m_vuln = (label_arr == 1), (label_arr == -1)
    feats = [f for f in META_FEATURES if f in df.columns]
    X_meta = (df[feats].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32))

    gdc_top = pd.to_numeric(df["global_dependent_packages_count"], errors="coerce")
    fill_path = OSV_DIR / "dependent_counts_fill.csv"
    if fill_path.exists():
        fill = pd.read_csv(fill_path)
        fmap = pd.to_numeric(fill.drop_duplicates("package", keep="last").set_index("package")["dependent_packages_count"], errors="coerce")
        gdc_fill = df["package"].map(fmap)
    else:
        gdc_fill = pd.Series(np.nan, index=df.index)
    global_deps = gdc_top.fillna(gdc_fill).fillna(0).to_numpy(dtype=np.float64)

    edges = pd.read_csv(DATA_DIR / "graph_edges.csv")
    pos = {p: i for i, p in enumerate(df["package"])}
    s_ser = edges["source"].map(pos); t_ser = edges["target"].map(pos)
    valid = s_ser.notna() & t_ser.notna()
    src = s_ser[valid].astype(np.int64).to_numpy(); tgt = t_ser[valid].astype(np.int64).to_numpy()
    conn_pkg = np.zeros(N, dtype=bool); conn_pkg[src] = True; conn_pkg[tgt] = True
    in_degree = np.bincount(tgt, minlength=N).astype(np.int64)
    out_degree = np.bincount(src, minlength=N).astype(np.int64)

    gs = build_graph_structures(src, tgt, N)
    n_scc, scc, lev, topo = gs["n_scc"], gs["scc"], gs["lev"], gs["topo"]
    scc_size = np.bincount(scc, minlength=n_scc).astype(np.float64); level_pkg = lev[scc]
    w_pop_scc = np.bincount(scc, weights=1.0 + np.log1p(np.maximum(global_deps, 0)), minlength=n_scc)
    drive = reach_weighted_sums(gs["inf_csr"].indptr, gs["inf_csr"].indices, topo[::-1],
                                np.stack([scc_size, w_pop_scc], axis=1))
    depnd = reach_weighted_sums(gs["dep_csr"].indptr, gs["dep_csr"].indices, topo, scc_size[:, None])
    driving_power = drive[scc, 0] - 1.0
    dependence_power = depnd[scc, 0] - 1.0

    conn_idx = np.flatnonzero(conn_pkg); nc = conn_idx.size
    remap = np.full(N, -1, np.int64); remap[conn_idx] = np.arange(nc)
    A_pd = sp.csr_matrix((np.ones(len(src)), (remap[src], remap[tgt])), shape=(nc, nc))
    AT = A_pd.T.tocsr(); outd = np.asarray(A_pd.sum(1)).ravel(); pr = np.full(nc, 1.0/nc)
    for _ in range(200):
        w = np.where(outd > 0, pr/np.where(outd > 0, outd, 1.0), 0.0)
        pr2 = 0.15/nc + 0.85*(AT@w + pr[outd == 0].sum()/nc)
        if np.abs(pr2-pr).sum() < 1e-12: pr = pr2; break
        pr = pr2
    pagerank = np.zeros(N); pagerank[conn_idx] = pr
    A_pkg = sp.csr_matrix((np.ones(len(src)), (src, tgt)), shape=(N, N)); h = np.ones(N)/N
    for _ in range(80):
        a = A_pkg.T@h; a /= max(np.linalg.norm(a), 1e-300); h2 = A_pkg@a; h2 /= max(np.linalg.norm(h2), 1e-300)
        if np.abs(h2-h).sum() < 1e-12: h = h2; break
        h = h2
    hits_hub, hits_auth = h, a

    l0_mask = conn_pkg & (level_pkg == 0)
    lge1_mask = conn_pkg & (level_pkg >= 1)

    # Stage-1 seeder (metadata RF, 5-fold OOF on manufacturers)
    l0 = np.flatnonzero(l0_mask); y_l0 = y[l0]; X_l0 = X_meta[l0]
    oof_l0 = np.zeros(len(l0))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_l0, y_l0):
        m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, class_weight="balanced")
        m.fit(X_l0[tr], y_l0[tr]); oof_l0[te] = m.predict_proba(X_l0[te])[:, 1]
    seed_full = np.zeros(N); seed_full[l0] = oof_l0

    # Noisy-OR propagation
    base_scc = np.zeros(n_scc); np.maximum.at(base_scc, scc, seed_full)
    risk = base_scc.copy(); dep_ip, dep_ix = gs["dep_csr"].indptr, gs["dep_csr"].indices
    for u in topo:
        a, b = dep_ip[u], dep_ip[u+1]
        if b > a:
            comp = (1.0-base_scc[u]) * np.prod(1.0 - ALPHA*risk[dep_ix[a:b]])
            risk[u] = float(np.clip(1.0-comp, 0.0, 1.0))
    prop_score = risk[scc]

    def scc_neighbor_mean(csr, vals_scc):
        ip, ix = csr.indptr, csr.indices; out = np.zeros(n_scc)
        for u in range(n_scc):
            a, b = ip[u], ip[u+1]
            if b > a: out[u] = vals_scc[ix[a:b]].mean()
        return out
    pred_risk = scc_neighbor_mean(gs["dep_csr"], risk); succ_risk = scc_neighbor_mean(gs["inf_csr"], risk)
    gp_scc = scc_neighbor_mean(gs["dep_csr"], pred_risk); ggp_scc = scc_neighbor_mean(gs["dep_csr"], gp_scc)
    pred_risk_pkg, succ_risk_pkg = pred_risk[scc], succ_risk[scc]; gp_pkg, ggp_pkg = gp_scc[scc], ggp_scc[scc]
    A_inf = sp.csr_matrix((np.ones(len(src), np.int8), (tgt, src)), shape=(N, N))
    hop = dijkstra(A_inf, directed=True, indices=l0, unweighted=True, min_only=True) if len(l0) else np.full(N, -1.0)
    hop = np.where(np.isfinite(hop), hop, -1.0)
    tele = np.zeros(nc); tele[remap[l0]] = oof_l0; tele = tele/tele.sum() if tele.sum() > 0 else np.full(nc, 1.0/nc)
    ppr = tele.copy()
    for _ in range(100):
        w = np.where(outd > 0, ppr/np.where(outd > 0, outd, 1.0), 0.0); ppr2 = 0.15*tele + 0.85*(AT@w + ppr[outd == 0].sum()*tele)
        if np.abs(ppr2-ppr).sum() < 1e-12: ppr = ppr2; break
        ppr = ppr2
    ppr_full = np.zeros(N); ppr_full[conn_idx] = ppr

    G_und = nx.Graph(); G_und.add_nodes_from(conn_idx.tolist()); G_und.add_edges_from(zip(src.tolist(), tgt.tolist()))
    clustering = np.zeros(N)
    for k, v in nx.clustering(G_und).items(): clustering[k] = v
    comms = louvain_communities(G_und, seed=SEED); comm_risk = np.zeros(N)
    for cm in comms:
        members = np.fromiter(cm, dtype=np.int64)
        comm_risk[members] = float((prop_score[members] > 0.1).mean()) if len(members) else 0.0
    rng_lm = np.random.default_rng(SEED); L = min(128, nc); landmarks = conn_idx[rng_lm.choice(nc, size=L, replace=False)]
    A_und = (A_pkg + A_pkg.T); A_und.data[:] = 1
    dists = dijkstra(A_und, directed=False, indices=landmarks, unweighted=True)
    with np.errstate(divide="ignore"): inv = np.where(dists > 0, 1.0/dists, 0.0)
    closeness = np.zeros(N); closeness[:] = np.nanmean(np.where(np.isfinite(inv), inv, 0.0), axis=0)

    X19 = np.column_stack([level_pkg, pagerank, in_degree, out_degree, in_degree+out_degree, clustering, closeness,
        hits_hub, hits_auth, np.log1p(driving_power), np.log1p(dependence_power),
        prop_score, hop, pred_risk_pkg, succ_risk_pkg, ppr_full, gp_pkg, ggp_pkg, comm_risk]).astype(np.float32)
    X_graph11 = np.column_stack([level_pkg, (~conn_pkg).astype(float), in_degree, out_degree, np.log1p(driving_power),
        np.log1p(dependence_power), scc_size[scc], pagerank, hits_hub, hits_auth, np.log1p(global_deps)]).astype(np.float32)
    X_union = np.hstack([X19, (~conn_pkg).astype(np.float32)[:, None], scc_size[scc].astype(np.float32)[:, None],
                         np.log1p(global_deps).astype(np.float32)[:, None]]).astype(np.float32)

    # risk time (compromised attack ts; vulnerable advisory ts)
    def first_ts(path, col, tcol):
        p = OSV_DIR / path
        if not p.exists() or p.stat().st_size == 0: return pd.Series(dtype="datetime64[ns]")
        dd = pd.read_csv(p, low_memory=False); dd[tcol] = pd.to_datetime(dd[tcol], errors="coerce", utc=True).dt.tz_localize(None)
        return dd.dropna(subset=[tcol]).groupby(col)[tcol].min()
    comp_ts = first_ts("malicious_all.csv", "package", "attack_timestamp")
    vuln_ts = first_ts("vulnerable_all.csv", "package", "first_advisory_timestamp")
    ct = df["package"].map(comp_ts); vt = df["package"].map(vuln_ts)
    risk_time = ct.where(pd.Series(m_comp), vt.where(pd.Series(m_vuln), pd.NaT)).to_numpy()
    at = pd.Series(risk_time)

    # prospective split at fixed 2026-01-01
    pos_t = at.notna().to_numpy() & (y == 1); pos_idx = np.flatnonzero(pos_t); at_pos = at.iloc[pos_idx].to_numpy()
    train_pos = pos_idx[at_pos < CUT.to_datetime64()]; test_pos = pos_idx[at_pos >= CUT.to_datetime64()]
    rng = np.random.default_rng(SEED)
    neg_idx = np.flatnonzero(y != 1); pn = rng.permutation(neg_idx); hf = len(pn)//2
    neg_tr = np.sort(pn[:hf]); neg_te = np.sort(pn[hf:])
    tr_idx = np.concatenate([train_pos, neg_tr]); ytr = np.r_[np.ones(len(train_pos)), np.zeros(len(neg_tr))].astype(int)

    # ---- train the three prospective detectors ----
    DET = {"Metadata (56)": X_meta, "Graph-11": X_graph11, "Union-22": X_union}
    models, scores = {}, {}
    for name, Xf in DET.items():
        m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, class_weight="balanced")
        m.fit(Xf[tr_idx], ytr)
        models[name] = m
        scores[name] = m.predict_proba(Xf)[:, 1]     # clean score for every node

    # ---- greedy evasion of the metadata detector (nb08 attack), attacker = the compromised pkg ----
    meta_m = models["Metadata (56)"]
    ref = np.median(X_meta[tr_idx][ytr == 0], axis=0)                 # benign target values
    medmal = np.median(X_meta[tr_idx][ytr == 1], axis=0)
    iqr = np.subtract(*np.percentile(X_meta[tr_idx], [75, 25], axis=0)) + 1.0
    order_meta = np.argsort(-np.abs(medmal - ref) / iqr)             # attack most-discriminative first

    def greedy_evade(model, Xatt, ref, order):
        Xc = Xatt.copy(); cur = model.predict_proba(Xc)[:, 1]
        for j in order:
            tr = Xc.copy(); tr[:, j] = ref[j]; new = model.predict_proba(tr)[:, 1]; better = new < cur
            Xc[better, j] = ref[j]; cur = np.where(better, new, cur)
        return Xc

    # test pool for percentile ranking: test positives + held-out negatives (clean metadata)
    te_idx = np.concatenate([test_pos, neg_te])
    meta_pool_clean = scores["Metadata (56)"][te_idx]                # ranking backdrop stays clean
    union_pool = scores["Union-22"][te_idx]
    graph_pool = scores["Graph-11"][te_idx]
    pkg_to_i = {p: i for i, p in enumerate(df["package"])}

    print(f"\n{'='*70}\n{eco.upper()}: prospective cutoff {CUT.date()} | train risky {len(train_pos)} "
          f"| test risky {len(test_pos)} | test pool {len(te_idx)}  ({time.time()-t0:.0f}s)")
    rows = []
    for p in CASE[eco]:
        if p not in pkg_to_i:
            print(f"  [skip] {p} not in data"); continue
        i = pkg_to_i[p]
        # evade this single package's metadata against the deployed metadata model
        x_ev = greedy_evade(meta_m, X_meta[i:i+1].copy(), ref, list(order_meta))
        p_meta_ev = float(meta_m.predict_proba(x_ev)[:, 1][0])
        p_meta_cl = float(scores["Metadata (56)"][i]); p_union = float(scores["Union-22"][i]); p_graph = float(scores["Graph-11"][i])
        pct = lambda s, pool: round(float((pool < s).mean()) * 100, 1)
        rows.append({
            "package": p, "date": str(pd.Timestamp(risk_time[i]).date()),
            "deps": int(global_deps[i]),
            "meta_clean_P": round(p_meta_cl, 3), "meta_clean_pct": pct(p_meta_cl, meta_pool_clean),
            "meta_evaded_P": round(p_meta_ev, 3), "meta_evaded_pct": pct(p_meta_ev, meta_pool_clean),
            "union_P": round(p_union, 3), "union_pct": pct(p_union, union_pool),
            "graph_P": round(p_graph, 3), "graph_pct": pct(p_graph, graph_pool),
        })
    res = pd.DataFrame(rows)
    pd.set_option("display.width", 240, "display.max_columns", 40)
    print(res.to_string(index=False))
    out_dir = ROOT / "data" / "output" / eco / "08_case_study_2026"
    out_dir.mkdir(parents=True, exist_ok=True)
    res.to_csv(out_dir / "case_study_2026.csv", index=False)
    return res

## 4. Run both ecosystems and assemble the case-study table

In [4]:
if ECO in CASE:
    res = run_eco(ECO)
    res
else:
    print(f"{ECO}: no case-study packages -- skipped")



NPM: prospective cutoff 2026-01-01 | train risky 3595 | test risky 1488 | test pool 49620  (168s)
                       package       date   deps  meta_clean_P  meta_clean_pct  meta_evaded_P  meta_evaded_pct  union_P  union_pct  graph_P  graph_pct
                         axios 2026-03-31 163723         0.737            99.7          0.043             41.4    0.847       97.8    0.736       97.5
               react-hook-form 2026-04-18   8172         0.367            97.0          0.033             35.6    0.390       95.0    0.458       95.4
                        hls.js 2026-04-23   1045         0.680            99.6          0.057             48.5    0.243       89.4    0.086       41.1
          @mistralai/mistralai 2026-05-11    850         0.110            69.6          0.017             24.1    0.563       97.1    0.462       95.5
                      node-ipc 2026-05-14    421         0.267            93.5          0.003             10.1    0.207       87.5    0.111       